# MERIT-ASU Microwave Imaging Pipeline
Run this notebook to reconstruct a 3D microwave image from S-parameter data.

In [ ]:
import sys
import os
# Ensure Python finds our local modules
sys.path.append(os.path.abspath('./python_colab'))

import matplotlib.pyplot as plt
from python_colab.ain_shams.tools.load_data_asu import load_data_asu
from python_colab.mimt.process import compute_beamform
from python_colab.mimt.visualization import display_2d, display_3d
import python_colab.mimt.manage_data as md

In [ ]:
# Set Configuration Parameters
params = {
    'ROI': 0.075,
    'slice': 0.03,
    'resolution': 2.5e-3,
    'relative_permittivity': 1.0,
    'background_subtraction': 1,
    'av_sub_cr': 1,
    'svd_cr': 0,
    'method': 'DAS',
    'feed_permittivity': 1.0,
    'feed_d': 15e-3,
    'gap': 2e-3,
    'f_start': 1e9,
    'f_end': 4e9,
    'freq_step': 1
}

data_opt = 0
conf_pol = 12
channels_mode = 3
scale_factor = -40

In [ ]:
# Load and Process Data
# NOTE: Make sure your 5 CSV files are uploaded into 'python_colab/ain_shams/data/'!
try:
    scan2, scan1, freqs, sensors_loc, ch_names = load_data_asu(data_opt, conf_pol, channels_mode, data_dir='python_colab/ain_shams/data')
    
    if np.sum(scan2) != 0:
        scan2 = md.scale_reflections(scan2, ch_names, scale_factor)
        scan1 = md.scale_reflections(scan1, ch_names, scale_factor)
        
        params['scan2'] = scan2
        params['scan1'] = scan1
        params['frequencies'] = freqs
        params['sensors_locations'] = sensors_loc
        params['channel_names'] = ch_names
        
        # Execute Beamformer
        img, tumor_x, tumor_y, grid_pts = compute_beamform(params)
        print(f"Estimated Tumor Location: X={tumor_x:.2f}mm, Y={tumor_y:.2f}mm")
except Exception as e:
    print("Error:", e)
    print("Did you forget to upload your data CSVs?")

In [ ]:
# Visualize 2D Slice
if 'img' in locals():
    plt.figure(figsize=(8, 6))
    display_2d(img, grid_pts * 1e3, 2, params['slice'] * 1e3, params['resolution'])
    plt.show()

In [ ]:
# Visualize 3D Volume
if 'img' in locals():
    display_3d(img, grid_pts * 1e3, sensors_loc * 1e3, 0.8)
    plt.show()